# Laboratorio inicial de Machine Learning con Iris

## Objetivo del laboratorio

Este laboratorio tiene como finalidad validar un entorno básico de Machine Learning en una máquina local, utilizando Python y librerías estándar del ecosistema de ciencia de datos, a través de un caso de clasificación supervisada sencillo.

En particular, el laboratorio busca:

- comprobar que el entorno de ejecución está correctamente configurado,
- familiarizarse con un flujo básico de trabajo de ML,
- comprender las etapas esenciales de preparación, entrenamiento y evaluación de un modelo,
- y validar que la máquina puede ejecutar cargas livianas de análisis y entrenamiento sin problemas operativos.

El caso elegido utiliza el dataset **Iris**, ampliamente usado en aprendizaje automático introductorio, por tratarse de un conjunto pequeño, limpio y bien conocido.

## Flujo general del laboratorio

El notebook sigue una secuencia estándar de Machine Learning supervisado:

1. importar librerías,
2. cargar el dataset,
3. estructurar variables predictoras y variable objetivo,
4. dividir los datos en entrenamiento y prueba,
5. entrenar un modelo,
6. generar predicciones,
7. evaluar el desempeño,
8. analizar la importancia relativa de las variables.

Este flujo representa una versión mínima pero técnicamente correcta de un pipeline de clasificación.

## 1. Importación de librerías

En este bloque se importan las dependencias necesarias para trabajar con datos, entrenar un modelo de clasificación y evaluar sus resultados.

### Justificación técnica

Toda implementación de Machine Learning necesita una base de herramientas especializadas. En este caso:

- `pandas` se usa para manipular datos tabulares con estructura tipo dataframe.
- `numpy` es la base numérica sobre la que operan muchas librerías científicas en Python.
- `load_iris` permite cargar un dataset estándar ya preparado.
- `train_test_split` divide los datos en subconjuntos de entrenamiento y prueba.
- `RandomForestClassifier` implementa un algoritmo de clasificación basado en bosques aleatorios.
- `accuracy_score` y `classification_report` permiten medir el desempeño del modelo.

Este bloque es necesario porque define la capa de herramientas sobre la cual se apoya todo el laboratorio.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## 2. Carga del dataset y definición de variables

En este bloque se carga el dataset Iris y se lo separa en dos componentes fundamentales del aprendizaje supervisado:

- `X`: variables predictoras o *features*,
- `y`: variable objetivo o *target*.

### Justificación técnica

En problemas de aprendizaje supervisado siempre se parte de dos elementos:

- una matriz de entrada con atributos observables,
- y una salida esperada que el modelo debe aprender a predecir.

En este caso, el dataset contiene mediciones físicas de flores, como largo y ancho de sépalos y pétalos. El objetivo es clasificar cada observación en una de tres especies.

### Por qué se usa `pandas`

Aunque `scikit-learn` puede trabajar con arreglos de NumPy, convertir los datos a `DataFrame` y `Series` mejora la legibilidad, preserva nombres de columnas y facilita el análisis exploratorio.

La instrucción `X.head(), y.head()` permite realizar una inspección rápida para verificar que los datos se hayan cargado correctamente.

**Cargamos todos los datos**

In [2]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

X.head(), y.head()

(   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
 0                5.1               3.5                1.4               0.2
 1                4.9               3.0                1.4               0.2
 2                4.7               3.2                1.3               0.2
 3                4.6               3.1                1.5               0.2
 4                5.0               3.6                1.4               0.2,
 0    0
 1    0
 2    0
 3    0
 4    0
 Name: target, dtype: int64)

## 3. División del dataset en entrenamiento y prueba

En este bloque se divide el conjunto de datos en dos subconjuntos:

- **training set**: se utiliza para entrenar el modelo,
- **test set**: se utiliza para evaluar su capacidad de generalización.

### Justificación técnica

Uno de los errores más comunes en Machine Learning es evaluar un modelo con los mismos datos con los que fue entrenado. Eso puede producir resultados artificialmente altos y dar una falsa sensación de desempeño.

Separar los datos en entrenamiento y prueba permite medir si el modelo realmente aprendió patrones generalizables y no simplemente memorizó ejemplos.

### Parámetros utilizados

- `test_size=0.2`: reserva el 20% de los datos para evaluación.
- `random_state=42`: fija la semilla aleatoria para que la partición sea reproducible.

La reproducibilidad es una buena práctica técnica, especialmente en laboratorios, documentación y comparación de resultados.

**Separamos los datos en train y test**

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4. Entrenamiento del modelo

En este bloque se crea y entrena un modelo de tipo **Random Forest**.

### ¿Qué es Random Forest?

Random Forest es un algoritmo de ensamble basado en múltiples árboles de decisión. Cada árbol aprende patrones a partir de subconjuntos de datos y variables, y la predicción final surge de la agregación de todos ellos.

### Justificación técnica de la elección

Se eligió este algoritmo porque:

- funciona bien en datasets pequeños y medianos,
- requiere poco preprocesamiento,
- es robusto frente a ruido,
- reduce el sobreajuste respecto de un árbol individual,
- y ofrece interpretabilidad básica a través de la importancia de variables.

### Parámetros utilizados

- `n_estimators=100`: define la cantidad de árboles del bosque.
- `random_state=42`: asegura reproducibilidad en el proceso de entrenamiento.

El método `.fit()` ajusta internamente el modelo a los datos de entrenamiento, aprendiendo la relación entre las variables de entrada y la clase objetivo.

**Entrenamos el modelo**

In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 5. Predicción y evaluación del modelo

En este bloque se generan predicciones sobre el conjunto de prueba y se calculan métricas para evaluar el desempeño del modelo.

### Justificación técnica

Una vez entrenado el modelo, el siguiente paso es medir qué tan bien predice datos no vistos previamente. Para eso se utiliza `X_test`, que contiene observaciones excluidas del entrenamiento.

Las predicciones generadas se comparan con los valores reales `y_test`.

### Métricas utilizadas

#### Accuracy

La *accuracy* mide la proporción de predicciones correctas sobre el total de observaciones evaluadas.

$$
Accuracy = \frac{Predicciones\ correctas}{Total\ de\ predicciones}
$$

Es una métrica útil cuando las clases están relativamente equilibradas, como en el dataset Iris.

#### Classification report

El reporte de clasificación agrega métricas por clase:

- **precision**: de todas las observaciones predichas como una clase, cuántas eran correctas,
- **recall**: de todas las observaciones reales de una clase, cuántas fueron detectadas correctamente,
- **f1-score**: media armónica entre precision y recall,
- **support**: cantidad de observaciones reales de cada clase.

Este bloque representa el momento en que el modelo deja de ser una estructura entrenada y pasa a ser evaluado objetivamente.

**evalúa accuracy** 

In [5]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print()
print(classification_report(y_test, y_pred))

Accuracy: 1.0000

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## 6. Importancia de variables

En este bloque se extrae la importancia relativa de cada variable según el modelo entrenado.

### Justificación técnica

En muchos problemas no alcanza con predecir bien. También es valioso entender qué variables tienen mayor peso en la decisión del modelo.

Random Forest permite calcular una medida interna de importancia basada en cuánto contribuye cada variable a reducir la impureza en los árboles del bosque.

### Utilidad práctica

Este análisis permite:

- obtener una primera interpretación del modelo,
- identificar variables más influyentes,
- detectar posibles features con bajo aporte,
- y comprender mejor la estructura del problema.

### Limitación importante

La importancia de variables no debe interpretarse como causalidad. Indica relevancia predictiva dentro del modelo, no una relación causal real entre variables y resultado.

**muestra importancia de variables**

In [6]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance

petal length (cm)    0.439994
petal width (cm)     0.421522
sepal length (cm)    0.108098
sepal width (cm)     0.030387
dtype: float64

## Conclusiones técnicas del laboratorio

Este laboratorio permite verificar que el entorno de Machine Learning es operativo y que la máquina puede ejecutar un flujo básico completo de clasificación supervisada.

Desde el punto de vista conceptual, el notebook resume las etapas esenciales de un pipeline de Machine Learning:

- definición de variables de entrada y salida,
- separación entre entrenamiento y evaluación,
- entrenamiento de un modelo supervisado,
- medición objetiva del desempeño,
- e interpretación inicial de resultados.

Aunque se trata de un caso simple, constituye una base sólida para evolucionar hacia experimentos más realistas y reproducibles.

## Conclusiones técnicas del laboratorio

Este laboratorio permite verificar que el entorno de Machine Learning es operativo y que la máquina puede ejecutar un flujo básico completo de clasificación supervisada.

Desde el punto de vista conceptual, el notebook resume las etapas esenciales de un pipeline de Machine Learning:

- definición de variables de entrada y salida,
- separación entre entrenamiento y evaluación,
- entrenamiento de un modelo supervisado,
- medición objetiva del desempeño,
- e interpretación inicial de resultados.

Aunque se trata de un caso simple, constituye una base sólida para evolucionar hacia experimentos más realistas y reproducibles.